In [1]:
from tqdm import tqdm
import numpy as np

import common

2024-03-21 15:02:13.724437: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-21 15:02:13.724457: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-21 15:02:13.725114: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-03-21 15:02:13.728858: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-21 15:02:14.371078: W tensorflow/compiler/tf2

In [2]:
RANDOM_SEED = 42
TRAIN_PR = 0.9
TEST_PR = 0.1

In [3]:
ds1 = common.load_dataset("data/training_25k_1.tfrecord", common.parse_feature)
ds4 = common.load_dataset("data/training_25k_4.tfrecord", common.parse_feature)

2024-03-21 15:02:15.187481: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-03-21 15:02:15.205137: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-03-21 15:02:15.205270: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [4]:
all_filenames = []
for ds  in [ds1, ds4]:
    filenames = []
    for example in tqdm(ds):
        filenames.append((example['image/filename'].numpy()).decode('utf-8'))
    all_filenames.append(np.array(filenames))

25000it [00:21, 1158.92it/s]
25001it [00:22, 1121.62it/s]


In [5]:
train_n = int(all_filenames[1].shape[0] * TRAIN_PR)
test_n = int(all_filenames[1].shape[0] * TEST_PR)

idxs = np.arange(all_filenames[1].shape[0])
np.random.seed(RANDOM_SEED)
np.random.shuffle(idxs)

train_names = all_filenames[1][idxs[:train_n]]
val_names = all_filenames[1][idxs[train_n:]]

filenames_available_for_test = np.setdiff1d(*all_filenames)
idxs = np.arange(filenames_available_for_test.shape[0])
np.random.seed(RANDOM_SEED)
np.random.shuffle(idxs)

test_names = filenames_available_for_test[idxs[:test_n]]

np.savetxt("data/train.txt", train_names, "%s")
np.savetxt("data/val.txt", val_names, "%s")
np.savetxt("data/test.txt", test_names, "%s")

train_names.shape[0], val_names.shape[0], test_names.shape[0]

(22500, 2501, 2500)

In [6]:
len(set(train_names).intersection(val_names)), len(set(train_names).intersection(test_names))

(0, 0)